In [1]:
%cd ..

/home/bhchen/LearnKalmanGain


In [2]:
import os
import glob
from PIL import Image

def create_gif_robust(image_folder, file_pattern, gif_path, duration=100, max_value=None):
    """
    Function:
        Creates a GIF from images matching a pattern with a numerical wildcard.
        This version is robust and does not depend on any keywords like 'timestep'.
        It identifies the number by what the '*' in the pattern matches.
    Input:
        image_folder (str): The path to the folder containing the images.
        file_pattern (str): Filename pattern with one '*' as a wildcard for a number.
                            Example: "frame_*_render.png"
        gif_path (str): The path to save the output GIF file.
        duration (int): Duration (in milliseconds) for each frame.
        max_value (int, optional): The maximum value of the wildcard part to include.
                                If None, all matched images will be used.
    Output:
        None
    """
    if file_pattern.count('*') != 1:
        print("Error: The file_pattern must contain exactly one '*' wildcard.")
        return

    # Dynamically determine the prefix and suffix from the pattern
    prefix, suffix = file_pattern.split('*')
    
    search_path = os.path.join(image_folder, file_pattern)
    all_filenames = glob.glob(search_path)
    
    if not all_filenames:
        print(f"Error: No images found in '{image_folder}' matching '{file_pattern}'.")
        return

    files_with_values = []
    for f_path in all_filenames:
        basename = os.path.basename(f_path)
        # Extract the part of the filename that corresponds to the wildcard
        if basename.startswith(prefix) and basename.endswith(suffix):
            try:
                # Remove prefix and suffix to get the numerical part
                value_str = basename[len(prefix):-len(suffix)]
                value = int(value_str)
                files_with_values.append({'value': value, 'path': f_path})
            except (ValueError, IndexError):
                # Ignore files where the wildcard part is not a valid integer
                print(f"Warning: Could not extract a valid number from '{basename}'. Skipping.")
                continue

    if not files_with_values:
        print("Error: Found matching files, but could not extract numbers from any of them.")
        return

    # Sort the files based on the extracted numerical value
    files_with_values.sort(key=lambda item: item['value'])

    # Filter by max_value if provided
    if max_value is not None:
        print(f"Filtering frames to include values up to {max_value}.")
        files_with_values = [item for item in files_with_values if item['value'] <= max_value]

    if not files_with_values:
        print(f"Error: After filtering, no images remained with a value <= {max_value}.")
        return
        
    final_filenames = [item['path'] for item in files_with_values]
    print(f"Creating GIF with {len(final_filenames)} frames...")

    images = [Image.open(fn) for fn in final_filenames]
    images[0].save(
        gif_path,
        save_all=True,
        append_images=images[1:],
        duration=duration,
        loop=0
    )
    print(f"GIF saved successfully at: {gif_path}")

In [6]:

# --- CONFIGURE YOUR SETUP HERE ---

dataset = 'rossler'
image_directory = f"save/{dataset}_pf_vis" 
traj_index = 1
pattern = f"sigma_y1.0_batch64_len500_pfN1000000_timestep*_42_{traj_index}_fixed.png"

frame_duration_ms = 75
max_timestep_to_include = 500 

output_gif_file = f"save/{dataset}_pf_vis/{dataset}_traj{traj_index}_{frame_duration_ms}ms_{max_timestep_to_include}timesteps.gif"

# --- END OF CONFIGURATION ---

# Run the function with your settings
create_gif_robust(
    image_folder=image_directory,
    file_pattern=pattern,
    gif_path=output_gif_file,
    duration=frame_duration_ms,
    max_value=max_timestep_to_include 
    )

Filtering frames to include values up to 500.
Creating GIF with 499 frames...
GIF saved successfully at: save/rossler_pf_vis/rossler_traj1_75ms_500timesteps.gif


In [10]:

dataset = 'lorenz96'
image_directory = f"save/{dataset}_pf_vis" 
pattern = f"sigma_y1.0_batch64_len500_pfN1000000_timestep*_42_{traj_index}_adaptive.png"

frame_duration_ms = 400
max_value = 64

output_gif_file = f"save/{dataset}_pf_vis/{dataset}_zoomin_{frame_duration_ms}ms_{max_timestep_to_include}timesteps.gif"

# Run the function with your settings
create_gif_robust(
    image_folder=image_directory,
    file_pattern=pattern,
    gif_path=output_gif_file,
    duration=frame_duration_ms,
    max_value=max_value 
)

Filtering frames to include values up to 64.
Creating GIF with 64 frames...
GIF saved successfully at: save/lorenz96_pf_vis/lorenz96_zoomin_400ms_500timesteps.gif
